### Structured Output

Structured outputs in LLMs are responses generated in a predefined format rather than free-form text. This makes the output easier for software to parse and use reliably.

#### Pydantic

Pydantic is a Python library for defining, validating, and parsing structured data using Python type hints.

Instead of manually checking data, you define a model and Pydantic automatically validates inputs.


In [27]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY")
)

##### Example - 1 

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description = "The title of the movie")
    year:int = Field(description = "The year the movie was released")
    director:str = Field(description = "The director of the movie: ")
    rating: float = Field(description="The movie's rating out of 10")


In [6]:
llm_with_structure = llm.with_structured_output(Movie)
llm_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000264166338C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002641686C440>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [10]:
llm_with_structure.invoke("Provide details about movie The Inglorius Bastards")

Movie(title='Inglourious Basterds', year=2009, director='Quentin Tarantino', rating=8.9)

##### Example - 2

Message output alongside parsed structure

In [11]:
llm_with_structure = llm.with_structured_output(Movie, include_raw=True)
llm_with_structure.invoke("Provide details about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb lists it at 8.8/10. So I have all the required parameters. I should structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title and director as strings, year as an integer, and rating as a number. No need for any other functions since the user just wants the movie details. Alright, time to put it all together.\n', 'tool_calls': [{'id': '8tybga8fg', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"tit

##### Example - 3

Nested Structured

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(description="Budget in million USD")

In [13]:
llm_with_structured = llm.with_structured_output(MovieDetails)

llm_with_structured.invoke("Provide details of the movie Inception")

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role="Dominick 'Dom' Toretto"), Actor(name='Joseph Gordon-Levitt', role="Brian O'Conner"), Actor(name='Ellen Page', role='Letty')], genres=['Action', 'Crime', 'Drama'], budget=160.0)

#### TypedDict

Type checking only. No runtime validation. Just describes the expected dictionary shape. Used by type checkers like mypy and IDEs

In [14]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The year of the movie"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie"]

In [18]:
llm_with_structured = llm.with_structured_output(MovieDict)

llm_with_structured.invoke("Provide details of the movie Avengers - Endgame")

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [22]:
llm.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Structured Outputs with Agents

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    "Contact Information for a person"
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email of the person")
    phone:str = Field(description = "The phone number of the person")

agent = create_agent(
    model = llm,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content" : "Extract contact info from John Doe, john@example.com, (555) 123-34242"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from John Doe, john@example.com, (555) 123-34242', additional_kwargs={}, response_metadata={}, id='cfbd6661-5790-456c-b4d8-1bec6a753503'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given details. The input is "John Doe, john@example.com, (555) 123-34242". I need to parse this into the ContactInfo function parameters: name, email, and phone.\n\nFirst, split the input by commas. The first part is the name, which is "John Doe". Then the email is "john@example.com". The phone number is "(555) 123-34242". Wait, the phone number has parentheses and a space, but the function doesn\'t specify any formatting. Should I keep it as-is or remove the formatting? The example in the function\'s parameters doesn\'t mention formatting, so maybe it\'s better to keep it as provided. The user didn\'t ask to format the phone number, just to extract it. S

In [29]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-34242')

### DataClasses

Dataclasses are a Python feature that automatically generates boilerplate code for classes that mainly store data.

In [31]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo():
    "Contact Information for a person"
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email of the person")
    phone:str = Field(description = "The phone number of the person")

agent = create_agent(
    model = llm,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content" : "Extract contact info from John Doe, john@example.com, (555) 123-34242"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from John Doe, john@example.com, (555) 123-34242', additional_kwargs={}, response_metadata={}, id='1698db01-5a8f-4f1e-8296-a1cf4bb92982'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given details: John Doe, john@example.com, (555) 123-34242. First, I need to identify which parts of this information correspond to the name, email, and phone number.\n\nThe name is straightforward—it\'s John Doe. The email is clearly john@example.com. The phone number is (555) 123-34242. But wait, the phone number format might need to be standardized. Typically, phone numbers are in the format (XXX) XXX-XXXX. Let me check the given number: (555) 123-34242. Hmm, the last part is 34242, which is five digits. That\'s not standard. Maybe it\'s a typo? But the user provided it as is, so I should probably keep it as given unless instructed otherwise. \n\nLooki